# Cell Matching × Segmentation: retrain the segmentation networks on Colab

Same code as the HPC package (`cellmatch_hpc.zip`). Colab's GPU replaces the SLURM jobs.

**Before you start**
1. Put the code and the data in the Google Drive folder `MyDrive/cellmatch/`: `cellmatch_hpc.zip` (or the
   unzipped `cellmatch_hpc` folder) and `Project_2_Dataset.zip` (or the unzipped `Project_2_Dataset` folder).
2. *Runtime → Change runtime type → GPU* (T4 is fine; L4/A100 on Colab Pro is faster).
3. Run the cells top to bottom. Pick **PATH** in the settings cell:

| PATH | what it does | approx. time on a T4 |
|---|---|---|
| `"fast"` | Fine-tunes the pretrained Cellpose **cyto3** model for ex-vivo on all three mice, keeps the current in-vivo model, and writes `submission.csv`. No validation: let the leaderboard judge it. | ~1 h |
| `"cv"` | Holds out one mouse (default db6b8b), compares ex-vivo models (`base` = current, `cyto3_x3`, `cyto3_x2`) with the full competition score, then trains the winner on all mice and writes `submission.csv`. | ~3 h |

Everything (checkpoints, cached model outputs, results) is saved to Drive under
`MyDrive/cellmatch/runs/`. **If Colab disconnects, reconnect and run all cells again**: finished
steps are skipped and training resumes from its last checkpoint (saved every 10 epochs).

In [ ]:
# ============================ SETTINGS ============================
PATH = "fast"                     # "fast" or "cv"  (see table above)
DRIVE_DIR = "/content/drive/MyDrive/cellmatch"

# fast path: which models to use for the final submission
FAST_IV, FAST_EX = "shipped", "cyto3_x3"     # "shipped" = current v3/v4 weights (no training)

# cv path: held-out mice and configurations to compare (see configs.py)
FOLDS = ["subject_db6b8b"]                    # add "subject_b2ba5e", "subject_5d294c" if you have time
IV_CONFIGS = ["base"]
EX_CONFIGS = ["base", "cyto3_x3", "cyto3_x2"]

SMOKE = False    # True = 3-epoch run on 2 regions per mouse, only to check the setup (writes runs_smoke/)

## 1. Setup (Drive, code, data, packages)

In [ ]:
import os, glob, json, zipfile, subprocess, shutil, sys
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)
# accepts either the .zip files or already-unzipped folders in DRIVE_DIR
for name in ["cellmatch_hpc", "Project_2_Dataset"]:
    z, d = os.path.join(DRIVE_DIR, name + ".zip"), os.path.join(DRIVE_DIR, name)
    if os.path.exists(z):
        with zipfile.ZipFile(z) as f:
            f.extractall(WORK)
    elif os.path.isdir(d):
        shutil.copytree(d, os.path.join(WORK, name), dirs_exist_ok=True)
    else:
        raise FileNotFoundError(f"put {name}.zip (or the unzipped {name} folder) in {DRIVE_DIR}")
CODE = os.path.dirname(glob.glob(f"{WORK}/**/train_seg_hpc.py", recursive=True)[0])
DATA = os.path.dirname(glob.glob(f"{WORK}/**/sample_submission.csv", recursive=True)[0])
RUNS = os.path.join(DRIVE_DIR, "runs_smoke" if SMOKE else "runs")
os.environ.update(CM_DATA=DATA, CM_RUNS=RUNS, CM_CACHE=f"{WORK}/cache", PYTHONUNBUFFERED="1",
                  CELLPOSE_LOCAL_MODELS_PATH=f"{WORK}/cellpose_models", CM_MAX_REGIONS="2" if SMOKE else "0")
os.makedirs(RUNS, exist_ok=True)
ntr = len(glob.glob(f"{DATA}/training/subject_*/region_*/exvivo.tif")); nte = len(glob.glob(f"{DATA}/hidden_test/subject_*/region_*/exvivo.tif"))
print("code:", CODE, "\ndata:", DATA, f"({ntr}/47 training, {nte}/29 test regions)", "\nruns:", RUNS)
assert ntr == 47 and nte == 29, "dataset incomplete - check Project_2_Dataset.zip"

In [ ]:
# Packages go into the environment; all heavy work runs in subprocesses, so no runtime restart is needed.
!pip -q install "cellpose==3.1.1.3" "scikit-learn==1.8.0"
!python -c "import torch, numpy, sklearn, cellpose; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| numpy', numpy.__version__, '| sklearn', sklearn.__version__)"

In [ ]:
def run(cmd):
    # run a script from the package folder, streaming its output (warnings filtered)
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=CODE, env=os.environ.copy(), text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        if ("warn" in line.lower() or "coo = torch" in line or "it/s]" in line or "empty masks" in line
                or "no seeds found" in line or not line.strip()):
            continue
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"command failed: {cmd}")

run("python -c \"import torch; assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > GPU'; print('GPU:', torch.cuda.get_device_name(0))\"")
EP = "--epochs 3" if SMOKE else ""

## 2a. Fast path
Trains the ex-vivo model on all three mice. With `FAST_IV = "shipped"` the in-vivo model is the
current one, so only the ex-vivo side changes compared with v4. It then caches both models' outputs on
the test set and builds `submission.csv`, using the v4 thresholds: in-vivo cellprob −1, ex-vivo
cellprob 0, pair threshold 0.1.

In [ ]:
def final_models(cfg_iv, cfg_ex):
    for mod, cfg in [("iv", cfg_iv), ("ex", cfg_ex)]:
        if cfg == "shipped":
            run(f"python infer_flows_hpc.py --mod {mod} --config shipped --fold full --split hidden_test --model weights/full_{mod}")
        else:
            run(f"python train_seg_hpc.py --mod {mod} --config {cfg} --fold full {EP}")
            run(f"python infer_flows_hpc.py --mod {mod} --config {cfg} --fold full --split hidden_test")

def make_submission(cfg_iv, cfg_ex, extra, name):
    out = os.path.join(DRIVE_DIR, name)
    run(f"python predict_test.py --iv-config {cfg_iv} --ex-config {cfg_ex} {extra} --out {out}")
    print("\nsubmission written to", out)

if PATH == "fast":
    final_models(FAST_IV, FAST_EX)
    make_submission(FAST_IV, FAST_EX, "--cp-iv -1 --cp-ex 0 --thr 0.1", f"submission_colab_{FAST_IV}_{FAST_EX}.csv")

## 2b. Cross-validation path
For each held-out mouse: train each configuration without that mouse, and cache its outputs on
the held-out mouse. Then `evaluate_cv.py` scores every in-vivo × ex-vivo × threshold setting with the
competition metric and writes `runs/best_config.json`.

In [ ]:
if PATH == "cv":
    for fold in FOLDS:
        for mod, cfgs in [("iv", IV_CONFIGS), ("ex", EX_CONFIGS)]:
            for cfg in cfgs:
                run(f"python train_seg_hpc.py --mod {mod} --config {cfg} --fold {fold} {EP}")
                run(f"python infer_flows_hpc.py --mod {mod} --config {cfg} --fold {fold}")
    cfgs = ",".join(sorted(set(IV_CONFIGS + EX_CONFIGS)))
    run(f"python evaluate_cv.py --workers {max(1, (os.cpu_count() or 2))} --configs {cfgs} --iv-top 1")
    print(json.load(open(os.path.join(RUNS, "best_config.json"))))

In [ ]:
# train the winner on all three mice and build the submission with its thresholds
if PATH == "cv":
    best = json.load(open(os.path.join(RUNS, "best_config.json")))
    final_models(best["iv_config"], best["ex_config"])
    make_submission(best["iv_config"], best["ex_config"],
                    f"--cp-iv {best['cp_iv']} --cp-ex {best['cp_ex']} --thr {best['thr']}",
                    f"submission_colab_cv_{best['iv_config']}_{best['ex_config']}.csv")

## 3. What to bring back
Download the following from `MyDrive/cellmatch/` into your `project 2` folder:
- the `submission_colab_*.csv` file(s)
- `runs/cv_stageA.csv`, `runs/cv_stageB.csv`, `runs/best_config.json` (cv path)
- the final weights `runs/<config>/<mod>/full/model` (needed to reproduce the submission for the report)